# SignSync — MLP Training (Google Colab)

**No GPU needed.** Free Colab CPU runtime finishes in ~10–15 minutes.

Run each cell top-to-bottom. After training you will download:
- `sign_language_mlp.onnx` — runtime model (ONNX, CPU inference)
- `sign_language_mlp.pt`   — PyTorch weights (for fine-tuning later)
- `class_names_mlp.json`   — ordered class labels

**Architecture:** 63-D MediaPipe landmarks → FC(256) → BN → ReLU → Dropout(0.3) → FC(128) → BN → ReLU → Dropout(0.2) → FC(33) → Softmax  
**33 classes:** A-Z (26) + space + del + help + danger + emergency + thumbs_down + ok_sign

## Step 1 — Install dependencies

In [ ]:
!pip install -q torch torchvision mediapipe scikit-learn tqdm onnx onnxruntime onnxscript kagglehub

## Step 2 — Kaggle credentials

Upload your `kaggle.json` (from https://www.kaggle.com/settings → API → Create New Token).
This gives access to the ASL Alphabet dataset (~1 GB, ~87 K images).

In [ ]:
# kagglehub reads credentials from ~/.kaggle/kaggle.json
# Upload your kaggle.json from https://www.kaggle.com/settings → API → Create New Token
from google.colab import files

uploaded = files.upload()  # select kaggle.json
import os
import shutil

if "kaggle.json" in uploaded:
    os.makedirs(os.path.expanduser("~/.kaggle"), exist_ok=True)
    shutil.copy("kaggle.json", os.path.expanduser("~/.kaggle/kaggle.json"))
    os.chmod(os.path.expanduser("~/.kaggle/kaggle.json"), 0o600)
    print("✅ Kaggle credentials configured")
    !python3 -c "import json; d=json.load(open('/root/.kaggle/kaggle.json')); print('   username:', d['username'])"
else:
    print("⚠️  No kaggle.json uploaded — download will fail in Step 5")

## Step 3 — Upload training script

Upload `train_standalone.py` from your project:
`src/app/core/ml/sign_model_mlp/train_standalone.py`

In [ ]:
import os

from google.colab import files

print("📂 Upload  train_standalone.py  from:")
print("   src/app/core/ml/sign_model_mlp/train_standalone.py")
uploaded = files.upload()

if "train_standalone.py" not in uploaded:
    raise FileNotFoundError("Wrong file uploaded — rename it to train_standalone.py and re-run this cell.")
size = os.path.getsize("train_standalone.py")
print(f"✅ train_standalone.py uploaded ({size:,} bytes)")

## Step 4 — Config (edit before running)

In [ ]:
# ── Edit these if needed ──────────────────────────────────────────────────
IMAGES_PER_LETTER = 500  # images per A-Z letter (500 → ~95% val accuracy)
SYNTHETIC_VARIATIONS = 500  # synthetic samples per emergency sign
EPOCHS = 60  # training epochs
LEARNING_RATE = 1e-3  # Adam learning rate

print("📋 Config:")
print(f"   Images/letter      : {IMAGES_PER_LETTER}")
print(f"   Synthetic/emergency: {SYNTHETIC_VARIATIONS}")
print(f"   Epochs             : {EPOCHS}")
print(f"   Learning rate      : {LEARNING_RATE}")
print()
print("ℹ️  Emergency signs (help/danger/emergency/thumbs_down/ok_sign)")
print("   use synthetic landmark templates — no extra dataset needed.")

## What's in this model

| Feature | Detail |
|---------|--------|
| Input | 63-D MediaPipe landmarks (21 pts × x,y,z — wrist-subtracted, scale-normalised) |
| Architecture | FC(256)→BN→ReLU→Drop(0.3) → FC(128)→BN→ReLU→Drop(0.2) → FC(33) |
| Parameters | ~85 K |
| Runtime | ONNX (no PyTorch at inference) — ~1 ms/frame on CPU |
| A-Z + space + del | Real landmarks extracted from Kaggle ASL alphabet dataset |
| Emergency signs | Synthetic: 500 augmented variations of hand-crafted templates |
| Optimizer | Adam (lr=1e-3, weight_decay=1e-4) |
| Scheduler | CosineAnnealingLR |
| Loss | CrossEntropyLoss with class weights (handles class imbalance) |

**Why synthetic for emergency signs?**  
No public dataset has these specific gestures in MediaPipe format. The synthetic approach
adds gaussian noise (σ=0.02) + scale jitter (±8%) to anatomically correct base poses — 
enough variation that the model generalises to real hands.

## Step 5 — Train!

In [ ]:
import glob
import importlib.util
import os

# Locate the training script
candidates = (
    glob.glob("train_standalone.py")
    + glob.glob("/content/train_standalone.py")
    + glob.glob("**/train_standalone.py", recursive=True)
)
if not candidates:
    raise FileNotFoundError(
        "train_standalone.py not found — did you run Step 3 (upload cell)?\n"
        "Upload src/app/core/ml/sign_model_mlp/train_standalone.py"
    )

script_path = os.path.abspath(candidates[0])
print(f"📄 Training script: {script_path}")

spec = importlib.util.spec_from_file_location("mlp_train", script_path)
module = importlib.util.module_from_spec(spec)
spec.loader.exec_module(module)

# Inject config from Step 4 into the module before training
module.IMAGES_PER_LETTER = IMAGES_PER_LETTER
module.SYNTHETIC_VARIATIONS = SYNTHETIC_VARIATIONS
module.EPOCHS = EPOCHS
module.LEARNING_RATE = LEARNING_RATE

module.main()

## (Optional) Re-export ONNX only

If training already finished (`.pt` file exists) but ONNX export failed,
run this cell instead of re-training everything.

In [ ]:
# Re-export .pt → .onnx without retraining
import glob
import importlib.util
import os

import torch

candidates = glob.glob("train_standalone.py") + glob.glob("/content/train_standalone.py")
script_path = os.path.abspath(candidates[0])

spec = importlib.util.spec_from_file_location("mlp_train", script_path)
module = importlib.util.module_from_spec(spec)
spec.loader.exec_module(module)

pt_path = "trained_model/sign_language_mlp.pt"
if not os.path.exists(pt_path):
    raise FileNotFoundError(f"{pt_path} not found — run Step 5 first")

# Rebuild model and load weights
model = module.SignLanguageMLP()
model.load_state_dict(torch.load(pt_path, map_location="cpu"))
module.export_model(model)
print("✅ Re-export complete — run Step 6 to download")

## Step 6 — Download trained model

In [ ]:
import os

from google.colab import files

for f in [
    "trained_model/sign_language_mlp.onnx",
    "trained_model/sign_language_mlp.pt",
    "trained_model/class_names_mlp.json",
]:
    if os.path.exists(f):
        files.download(f)
        print(f"⬇️  Downloaded {f}")
    else:
        print(f"⚠️  Not found: {f}")

print()
print("📁 Place downloaded files in:")
print("   src/app/core/ml/sign_model_mlp/trained_model/")
print()
print("🔄 Then restart Docker:")
print("   docker compose restart web")
print("   (no rebuild needed — files are volume-mounted)")